# Stratified Split 70/15/15 — Lung Cancer MRI Dataset

Dataset asli memiliki split `train`/`validate` bawaan Kaggle, tapi proposal skripsi ini memakai skema **70% train / 15% val / 15% test** yang stratified terhadap kelas. Notebook ini menggabungkan seluruh citra dari `train` + `validate` (berdasarkan manifest dari `01_eda.ipynb`), lalu membaginya ulang menjadi tiga split baru di folder `dataset_split/`.

> **Catatan:** ada nama file yang sama antara folder `train/cancer` dan `validate/cancer` pada dataset asli (371 nama duplikat), sehingga saat menyalin file, nama tujuan diberi prefix split asal (`train__...` / `validate__...`) agar tidak saling menimpa.


In [1]:
import shutil
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

MANIFEST_PATH = Path("../reports/dataset_manifest.csv")
OUTPUT_ROOT = Path("../dataset_split")
RANDOM_SEED = 42

manifest = pd.read_csv(MANIFEST_PATH)
print(f"Total citra dalam manifest: {len(manifest)}")
manifest["label"].value_counts()


Total citra dalam manifest: 3680


label
cancer       1874
no_cancer    1806
Name: count, dtype: int64

## 1. Split stratified 70/15/15

In [2]:
train_df, temp_df = train_test_split(
    manifest,
    test_size=0.30,
    stratify=manifest["label"],
    random_state=RANDOM_SEED,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=RANDOM_SEED,
)

train_df = train_df.copy()
train_df["new_split"] = "train"
val_df = val_df.copy()
val_df["new_split"] = "val"
test_df = test_df.copy()
test_df["new_split"] = "test"

split_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

print(split_df.groupby(["new_split", "label"]).size().unstack(fill_value=0))
print("\nProporsi split (%):")
print((split_df["new_split"].value_counts(normalize=True) * 100).round(2))


label      cancer  no_cancer
new_split                   
test          281        271
train        1312       1264
val           281        271

Proporsi split (%):
new_split
train    70.0
val      15.0
test     15.0
Name: proportion, dtype: float64


## 2. Salin file ke folder `dataset_split/`

In [3]:
for new_split in ["train", "val", "test"]:
    for label in ["cancer", "no_cancer"]:
        (OUTPUT_ROOT / new_split / label).mkdir(parents=True, exist_ok=True)

def dest_filename(row):
    return f"{row['split']}__{row['filename']}"

split_df["dest_filename"] = split_df.apply(dest_filename, axis=1)
split_df["dest_path"] = split_df.apply(
    lambda r: str(OUTPUT_ROOT / r["new_split"] / r["label"] / r["dest_filename"]), axis=1
)

for i, row in enumerate(split_df.itertuples(index=False), start=1):
    shutil.copy2(row.path, row.dest_path)
    if i % 500 == 0:
        print(f"{i}/{len(split_df)} file disalin...")

print("Selesai menyalin semua file.")


500/3680 file disalin...


1000/3680 file disalin...


1500/3680 file disalin...


2000/3680 file disalin...


2500/3680 file disalin...


3000/3680 file disalin...


3500/3680 file disalin...


Selesai menyalin semua file.


## 3. Verifikasi hasil split

In [4]:
verify_counts = {}
for new_split in ["train", "val", "test"]:
    for label in ["cancer", "no_cancer"]:
        folder = OUTPUT_ROOT / new_split / label
        verify_counts[(new_split, label)] = len(list(folder.iterdir()))

verify_df = pd.Series(verify_counts).unstack()
print(verify_df)
print("\nTotal file di dataset_split/:", verify_df.values.sum())
print("Total file di manifest asli:", len(manifest))
assert verify_df.values.sum() == len(manifest), "Jumlah file tidak cocok!"
print("Verifikasi OK — jumlah file cocok.")


       cancer  no_cancer
test      281        271
train    1312       1264
val       281        271

Total file di dataset_split/: 3680
Total file di manifest asli: 3680
Verifikasi OK — jumlah file cocok.


## 4. Simpan manifest split final

In [5]:
split_df.to_csv("../reports/dataset_split_manifest.csv", index=False)
print("Manifest split disimpan ke reports/dataset_split_manifest.csv")
split_df[["new_split", "label"]].value_counts().sort_index()


Manifest split disimpan ke reports/dataset_split_manifest.csv


new_split  label    
test       cancer        281
           no_cancer     271
train      cancer       1312
           no_cancer    1264
val        cancer        281
           no_cancer     271
Name: count, dtype: int64